# MC1 - Exploratory Data Analysis (EDA)

Análisis exploratorio del dataset del **VAST Challenge 2025 Mini-Challenge 1** (Influence and Creativity in the Music Industry).

Las gráficas están diseñadas con el **estilo visual de Melody Way** (paleta oscura tipo "galaxia" + acentos de color por género) para que se integren bien con la presentación.

Cada gráfica responde a una pregunta concreta sobre el dataset y se exporta como PNG transparente para pegar en las slides.

In [ ]:
import json
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import os

# Estilo Melody Way: fondo oscuro, texto claro, paleta vibrante
plt.rcParams.update({
    'figure.facecolor':  '#0b0d18',
    'axes.facecolor':    '#0b0d18',
    'savefig.facecolor': '#0b0d18',
    'axes.edgecolor':    '#3a3f5a',
    'axes.labelcolor':   '#e6e8f0',
    'xtick.color':       '#a8acc4',
    'ytick.color':       '#a8acc4',
    'text.color':        '#e6e8f0',
    'axes.titlecolor':   '#ffffff',
    'axes.titleweight':  'bold',
    'axes.titlesize':    14,
    'font.family':       'DejaVu Sans',
    'axes.grid':         True,
    'grid.color':        '#1f2238',
    'grid.linestyle':    '-',
    'grid.linewidth':    0.6,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

# Paleta Melody Way (tomada de los géneros visibles en Overview.jsx)
MW_PALETTE = [
    '#fbbf24',  # Oceanus Folk – dorado
    '#60a5fa',  # azul
    '#34d399',  # verde
    '#f87171',  # rojo coral
    '#c084fc',  # violeta
    '#fb923c',  # naranja
    '#22d3ee',  # cyan
    '#f472b6',  # rosa
    '#a3e635',  # lima
    '#facc15',  # amarillo
]

OUT_DIR = 'eda_charts'
os.makedirs(OUT_DIR, exist_ok=True)

INPUT_PATH = 'MC1_graph.json'  # ajustá si lo tenés en otro lado
with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

nodes = raw['nodes']
edges = raw['links']
print(f'Nodos: {len(nodes):,}  |  Edges: {len(edges):,}')

## 1. Composición del knowledge graph

**Pregunta:** ¿De qué está hecho el grafo? ¿Cuántas entidades hay de cada tipo?

In [ ]:
node_types = Counter(n['Node Type'] for n in nodes)
edge_types = Counter(e['Edge Type'] for e in edges)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Tipos de nodo (horizontal bar)
ax = axes[0]
labels = ['Person', 'Song', 'RecordLabel', 'Album', 'MusicalGroup']
vals = [node_types[k] for k in labels]
colors = ['#60a5fa', '#fbbf24', '#fb923c', '#c084fc', '#34d399']
bars = ax.barh(labels, vals, color=colors, edgecolor='none')
for b, v in zip(bars, vals):
    ax.text(v + max(vals)*0.01, b.get_y() + b.get_height()/2,
            f'{v:,}', va='center', color='#e6e8f0', fontsize=11)
ax.set_title('Tipos de nodo')
ax.set_xlim(0, max(vals) * 1.15)
ax.invert_yaxis()

# Tipos de edge
ax = axes[1]
items = edge_types.most_common()
lbls = [k for k, _ in items]
vs = [v for _, v in items]
# Colorear edges de influencia distinto
INFL = {'CoverOf', 'DirectlySamples', 'LyricalReferenceTo', 'InterpolatesFrom', 'InStyleOf'}
edge_colors = ['#fbbf24' if k in INFL else '#60a5fa' for k in lbls]
bars = ax.barh(lbls, vs, color=edge_colors, edgecolor='none')
for b, v in zip(bars, vs):
    ax.text(v + max(vs)*0.01, b.get_y() + b.get_height()/2,
            f'{v:,}', va='center', color='#e6e8f0', fontsize=10)
ax.set_title('Tipos de edge (azul = roles/sello, dorado = influencia)')
ax.set_xlim(0, max(vs) * 1.18)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/01_graph_composition.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Distribución temporal de los artworks

**Pregunta:** ¿Cómo se distribuye la producción musical a lo largo del tiempo?

Esto importa porque el eje radial de Melody Way usa el último año de release de cada artista. Saber el rango temporal nos dice cuánto "abre" la galaxia.

In [ ]:
artworks = [n for n in nodes if n['Node Type'] in ('Song', 'Album')]

def parse_year(v):
    try:
        return int(v)
    except (TypeError, ValueError):
        return None

songs_by_year  = Counter(parse_year(n.get('release_date')) for n in artworks if n['Node Type']=='Song')
albums_by_year = Counter(parse_year(n.get('release_date')) for n in artworks if n['Node Type']=='Album')
songs_by_year.pop(None, None)
albums_by_year.pop(None, None)

all_years = sorted(set(songs_by_year) | set(albums_by_year))
songs_v  = [songs_by_year.get(y, 0) for y in all_years]
albums_v = [albums_by_year.get(y, 0) for y in all_years]

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(all_years, songs_v,  label='Songs',  color='#fbbf24', edgecolor='none')
ax.bar(all_years, albums_v, bottom=songs_v, label='Albums', color='#c084fc', edgecolor='none')
ax.set_title(f'Releases por año ({min(all_years)}–{max(all_years)})')
ax.set_xlabel('Año')
ax.set_ylabel('Cantidad de artworks')
ax.legend(facecolor='#11142a', edgecolor='#3a3f5a', labelcolor='#e6e8f0')
ax.axvline(2025, color='#f87171', linestyle='--', alpha=0.6, linewidth=1)
ax.text(2025.4, max(np.array(songs_v)+np.array(albums_v))*0.92, 'Hoy', color='#f87171', fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/02_releases_per_year.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total artworks con año: {sum(songs_v)+sum(albums_v):,}')
print(f'Rango: {min(all_years)} – {max(all_years)}')

## 3. Distribución de géneros

**Pregunta:** ¿Cuáles son los géneros más representados? ¿Dónde se ubica Oceanus Folk?

Este chart resalta a Oceanus Folk porque es el género focal del challenge.

In [ ]:
genres = Counter(n.get('genre') for n in artworks)
genres.pop(None, None)
items = genres.most_common()

lbls = [k for k, _ in items]
vs   = [v for _, v in items]
colors = ['#fbbf24' if g == 'Oceanus Folk' else '#60a5fa' for g in lbls]

fig, ax = plt.subplots(figsize=(11, 8))
bars = ax.barh(lbls, vs, color=colors, edgecolor='none')
for b, v, g in zip(bars, vs, lbls):
    fontw = 'bold' if g == 'Oceanus Folk' else 'normal'
    ax.text(v + max(vs)*0.008, b.get_y() + b.get_height()/2,
            f'{v:,}', va='center', color='#e6e8f0', fontsize=9.5, fontweight=fontw)
ax.set_title('Artworks por género (Oceanus Folk resaltado)')
ax.set_xlabel('Cantidad de artworks')
ax.invert_yaxis()
ax.set_xlim(0, max(vs) * 1.10)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/03_genre_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Géneros únicos: {len(items)}')
print(f'Oceanus Folk: {genres["Oceanus Folk"]} artworks ({100*genres["Oceanus Folk"]/sum(vs):.1f}% del total)')

## 4. Edges de influencia: composición y peso

**Pregunta:** ¿Cuántas relaciones de influencia hay y cómo se reparten? Cada tipo tiene un peso heurístico distinto en el cálculo del score.

In [ ]:
INFLUENCE_WEIGHTS = {
    'CoverOf': 1.0, 'DirectlySamples': 0.9, 'LyricalReferenceTo': 0.8,
    'InterpolatesFrom': 0.7, 'InStyleOf': 0.6,
}
inf_counts = {k: edge_types[k] for k in INFLUENCE_WEIGHTS}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

# Pie chart - cantidad por tipo
ax = axes[0]
lbls = list(inf_counts.keys())
vs   = list(inf_counts.values())
pcols = ['#fbbf24', '#fb923c', '#f472b6', '#c084fc', '#60a5fa']
wedges, texts, autotexts = ax.pie(vs, labels=lbls, colors=pcols, autopct='%1.1f%%',
                                   startangle=90, wedgeprops={'edgecolor':'#0b0d18','linewidth':2},
                                   textprops={'color': '#e6e8f0', 'fontsize': 10})
for at in autotexts:
    at.set_color('#0b0d18')
    at.set_fontweight('bold')
ax.set_title(f'Distribución de los {sum(vs):,} edges de influencia')

# Bar chart de pesos
ax = axes[1]
weights = [INFLUENCE_WEIGHTS[k] for k in lbls]
bars = ax.bar(lbls, weights, color=pcols, edgecolor='none')
for b, w in zip(bars, weights):
    ax.text(b.get_x()+b.get_width()/2, w + 0.02, f'{w}', ha='center',
            color='#e6e8f0', fontweight='bold', fontsize=11)
ax.set_title('Pesos heurísticos por tipo (Melody Way)')
ax.set_ylabel('Peso')
ax.set_ylim(0, 1.2)
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/04_influence_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Notable vs no-notable

**Pregunta:** ¿Qué proporción de artworks son "notable"? Este atributo decide cómo se renderizan los nodos en el timeline (centro blanco vs vacío).

In [ ]:
notable = Counter(bool(n.get('notable')) for n in artworks)

fig, ax = plt.subplots(figsize=(7, 5))
vals = [notable[True], notable[False]]
lbls = ['Notable', 'No notable']
cols = ['#fbbf24', '#3a3f5a']
bars = ax.bar(lbls, vals, color=cols, edgecolor='none', width=0.55)
for b, v in zip(bars, vals):
    pct = 100 * v / sum(vals)
    ax.text(b.get_x()+b.get_width()/2, v + max(vals)*0.01,
            f'{v:,}\n({pct:.1f}%)', ha='center', color='#e6e8f0', fontsize=11)
ax.set_title('Artworks notable vs. no notable')
ax.set_ylabel('Cantidad')
ax.set_ylim(0, max(vals) * 1.15)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/05_notable.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Distribución de productividad de artistas

**Pregunta:** ¿Hay muchos artistas con pocas obras y pocos con muchas (long tail)?

Esto justifica el uso del eje radial: el offset dentro de cada banda en Melody Way representa la productividad.

In [ ]:
ROLE_TYPES = {'PerformerOf', 'ComposerOf', 'ProducerOf', 'LyricistOf'}
person_artworks = defaultdict(set)
for e in edges:
    if e['Edge Type'] in ROLE_TYPES:
        person_artworks[e['source']].add(e['target'])

counts = sorted([len(v) for v in person_artworks.values()], reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
ax = axes[0]
bins = np.arange(1, max(counts)+2) - 0.5
ax.hist(counts, bins=bins, color='#60a5fa', edgecolor='#0b0d18', linewidth=0.5)
ax.set_yscale('log')
ax.set_title('Distribución de artworks por artista (log)')
ax.set_xlabel('Cantidad de artworks contribuidos')
ax.set_ylabel('Cantidad de artistas (log)')
ax.set_xlim(0, max(counts)+1)

# Top 15 más productivos
ax = axes[1]
node_by_id = {n['id']: n for n in nodes}
top = sorted(person_artworks.items(), key=lambda x: -len(x[1]))[:15]
names = [node_by_id[k].get('name', str(k))[:25] for k, _ in top]
vs = [len(v) for _, v in top]
bars = ax.barh(names, vs, color='#fbbf24', edgecolor='none')
for b, v in zip(bars, vs):
    ax.text(v + max(vs)*0.01, b.get_y()+b.get_height()/2, str(v),
            va='center', color='#e6e8f0', fontsize=10)
ax.set_title('Top 15 artistas/grupos más productivos')
ax.invert_yaxis()
ax.set_xlim(0, max(vs)*1.10)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/06_productivity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mediana: {int(np.median(counts))}, P95: {int(np.percentile(counts, 95))}, Max: {max(counts)}')

## 7. Oceanus Folk en el tiempo: emisor vs receptor de influencia

**Pregunta:** ¿Cómo evolucionó Oceanus Folk? ¿Recibe más influencia o la emite?

Esta gráfica replica conceptualmente la Figura 3 del paper de Melody Way.

In [ ]:
INFL = {'CoverOf', 'DirectlySamples', 'LyricalReferenceTo', 'InterpolatesFrom', 'InStyleOf'}
node_by_id = {n['id']: n for n in nodes}

out_by_year = Counter()  # Oceanus Folk influencia a otros (source = OF)
in_by_year  = Counter()  # otros influencian a Oceanus Folk (target = OF)

for e in edges:
    if e['Edge Type'] not in INFL:
        continue
    src = node_by_id.get(e['source'])
    tgt = node_by_id.get(e['target'])
    if not src or not tgt:
        continue
    if src.get('Node Type') in ('Song','Album') and tgt.get('Node Type') in ('Song','Album'):
        # year = release del SOURCE (el que cita / hace cover / etc.)
        try:
            y = int(src.get('release_date'))
        except (TypeError, ValueError):
            continue
        if tgt.get('genre') == 'Oceanus Folk' and src.get('genre') != 'Oceanus Folk':
            in_by_year[y] += 1
        if src.get('genre') == 'Oceanus Folk' and tgt.get('genre') != 'Oceanus Folk':
            # OF emite -- pero el peso temporal lo acreditamos al año del SOURCE igual
            out_by_year[y] += 1

years = sorted(set(in_by_year) | set(out_by_year))
in_v  = [in_by_year.get(y, 0)  for y in years]
out_v = [out_by_year.get(y, 0) for y in years]

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(years, in_v,  alpha=0.45, color='#fbbf24', label='Influencia recibida (otros → Oceanus Folk)')
ax.plot(years, in_v, color='#fbbf24', linewidth=2)
ax.fill_between(years, out_v, alpha=0.30, color='#60a5fa', label='Influencia emitida (Oceanus Folk → otros)')
ax.plot(years, out_v, color='#60a5fa', linewidth=2)
ax.set_title('Oceanus Folk: dinámica de influencia en el tiempo')
ax.set_xlabel('Año del artwork que ejerce/recibe la influencia')
ax.set_ylabel('Cantidad de edges de influencia')
ax.legend(facecolor='#11142a', edgecolor='#3a3f5a', labelcolor='#e6e8f0', loc='upper left')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/07_oceanus_folk_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Top sellos por catálogo (Recorded vs Distributed)

**Pregunta:** ¿Cuáles son los sellos con más actividad? Esta vista justifica los nodos amarillos del timeline en Melody Way.

In [ ]:
label_rec = defaultdict(set)
label_dis = defaultdict(set)
for e in edges:
    if e['Edge Type'] == 'RecordedBy':
        if node_by_id[e['target']]['Node Type'] == 'RecordLabel':
            label_rec[e['target']].add(e['source'])
        elif node_by_id[e['source']]['Node Type'] == 'RecordLabel':
            label_rec[e['source']].add(e['target'])
    elif e['Edge Type'] == 'DistributedBy':
        if node_by_id[e['target']]['Node Type'] == 'RecordLabel':
            label_dis[e['target']].add(e['source'])
        elif node_by_id[e['source']]['Node Type'] == 'RecordLabel':
            label_dis[e['source']].add(e['target'])

all_labels = set(label_rec) | set(label_dis)
totals = sorted(
    [(lid, len(label_rec[lid]), len(label_dis[lid])) for lid in all_labels],
    key=lambda x: -(x[1] + x[2])
)[:15]

names = [node_by_id[lid]['name'][:30] for lid, _, _ in totals]
rec_v = [r for _, r, _ in totals]
dis_v = [d for _, _, d in totals]

fig, ax = plt.subplots(figsize=(11, 6.5))
y = np.arange(len(names))
ax.barh(y, rec_v, color='#fbbf24', label='Recorded',    edgecolor='none')
ax.barh(y, dis_v, left=rec_v, color='#60a5fa', label='Distributed', edgecolor='none')
for i, (r, d) in enumerate(zip(rec_v, dis_v)):
    ax.text(r+d + max(np.array(rec_v)+np.array(dis_v))*0.01, i, f'{r+d}',
            va='center', color='#e6e8f0', fontsize=10)
ax.set_yticks(y)
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_title('Top 15 sellos por volumen de catálogo')
ax.set_xlabel('Cantidad de artworks')
ax.legend(facecolor='#11142a', edgecolor='#3a3f5a', labelcolor='#e6e8f0', loc='lower right')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/08_top_labels.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Calidad de los datos: missing values y tipos extraños

**Pregunta:** ¿Hay campos faltantes o anomalías que la transformación tenga que tolerar?

In [ ]:
checks = {}
checks['Songs sin género']               = sum(1 for n in artworks if n['Node Type']=='Song'  and not n.get('genre'))
checks['Albums sin género']              = sum(1 for n in artworks if n['Node Type']=='Album' and not n.get('genre'))
checks['Artworks sin release_date']      = sum(1 for n in artworks if not n.get('release_date'))
checks['release_date no numérico']       = sum(1 for n in artworks if n.get('release_date') and parse_year(n['release_date']) is None)
checks['Persons con stage_name']         = sum(1 for n in nodes if n['Node Type']=='Person' and n.get('stage_name'))

# Edges raros
INFL = {'CoverOf', 'DirectlySamples', 'LyricalReferenceTo', 'InterpolatesFrom', 'InStyleOf'}
n_label_reverse = sum(1 for e in edges if e['Edge Type'] in {'RecordedBy','DistributedBy'}
                        and node_by_id[e['source']]['Node Type']=='RecordLabel')
n_label_producer = sum(1 for e in edges if e['Edge Type']=='ProducerOf'
                          and node_by_id[e['source']]['Node Type']=='RecordLabel')
n_artist_inf = sum(1 for e in edges if e['Edge Type'] in INFL
                       and node_by_id[e['target']]['Node Type'] in ('Person','MusicalGroup'))
checks['Edges label→artwork (RecordedBy/DistributedBy)'] = n_label_reverse
checks['Edges label→artwork (ProducerOf)']                = n_label_producer
checks['Edges de influencia artista→artista']             = n_artist_inf

fig, ax = plt.subplots(figsize=(11, 5.5))
lbls = list(checks.keys())
vs   = list(checks.values())
colors = ['#34d399' if v == 0 else '#f87171' if v < 50 else '#fb923c' for v in vs]
bars = ax.barh(lbls, vs, color=colors, edgecolor='none')
for b, v in zip(bars, vs):
    ax.text(v + max(vs)*0.01 if vs else 0.1, b.get_y()+b.get_height()/2,
            f'{v:,}', va='center', color='#e6e8f0', fontsize=10)
ax.set_title('Anomalías y casos especiales en el dataset')
ax.set_xlabel('Cantidad de registros')
ax.invert_yaxis()
if max(vs) > 0:
    ax.set_xlim(0, max(vs)*1.12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/09_data_quality.png', dpi=150, bbox_inches='tight')
plt.show()

for k, v in checks.items():
    print(f'  {k}: {v}')

## 10. Resumen ejecutivo (data card)

Una sola imagen con los KPIs clave del dataset, lista para pegar en una slide.

In [ ]:
kpis = [
    ('Nodos totales',           f"{len(nodes):,}",     '#fbbf24'),
    ('Edges totales',           f"{len(edges):,}",     '#60a5fa'),
    ('Artistas (Person+Group)', f"{node_types['Person']+node_types['MusicalGroup']:,}", '#34d399'),
    ('Artworks (Song+Album)',   f"{node_types['Song']+node_types['Album']:,}",          '#c084fc'),
    ('Sellos discográficos',    f"{node_types['RecordLabel']:,}",                        '#fb923c'),
    ('Géneros únicos',          f"{len(set(n.get('genre') for n in artworks if n.get('genre')))}", '#f472b6'),
    ('Edges de influencia',     f"{sum(edge_types[k] for k in INFL):,}",                 '#22d3ee'),
    ('Rango temporal',          f"{min(all_years)}–{max(all_years)}",                    '#a3e635'),
]

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.set_xlim(0, 4); ax.set_ylim(0, 2)
ax.axis('off')
ax.set_title('MC1 dataset · resumen', fontsize=18, pad=15)

for i, (lbl, val, col) in enumerate(kpis):
    col_idx = i % 4
    row_idx = 1 - (i // 4)
    cx = col_idx + 0.5
    cy = row_idx + 0.5
    # 'tarjeta'
    ax.add_patch(plt.Rectangle((col_idx+0.06, row_idx+0.10), 0.88, 0.82,
                                facecolor='#11142a', edgecolor=col, linewidth=1.5))
    ax.text(cx, cy+0.18, val, ha='center', va='center',
            fontsize=22, fontweight='bold', color=col)
    ax.text(cx, cy-0.22, lbl, ha='center', va='center',
            fontsize=10.5, color='#a8acc4')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/10_summary_card.png', dpi=150, bbox_inches='tight')
plt.show()

## Listado de PNGs generados

Todos quedan en la carpeta `eda_charts/` listos para pegar en las slides.

In [ ]:
for f in sorted(os.listdir(OUT_DIR)):
    print(' ', f)